# 🎯 Notebook 05 — Model Training (Classification)

**Purpose**: Train binary classifiers for `over_temp_flag` and handle `over_voltage_flag` with rule-based fallback.

**Theory**:
- Lecture 04: Logistic Regression — sigmoid function, decision boundary
- Lecture 06: SVM with RBF kernel — non-linear boundaries via kernel trick
- Lecture 07: XGBoost Classifier — sequential boosting with regularization
- Lecture 03: **Recall is primary** when False Negatives are costly (missing overheating = dangerous)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, sys, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
os.makedirs('../plots', exist_ok=True)
sys.path.insert(0, '..')
from src.feature_engineer import engineer_all_features
print('Ready.')

In [ ]:
# Load and prepare data (same pipeline as NB04)
df = pd.read_csv('../data/nev_battery_charging.csv').drop(columns=['timestamp']).drop_duplicates()
n = len(df); t_end, v_end = int(n*0.70), int(n*0.85)
targets = ['cycle_degradation','over_temp_flag','over_voltage_flag']
df_train, df_val, df_test = df.iloc[:t_end].copy(), df.iloc[t_end:v_end].copy(), df.iloc[v_end:].copy()
baseline_ir = df_train['internal_resistance'].iloc[0]

df_train = engineer_all_features(df_train, baseline_ir, ['battery_temp'])
df_val = engineer_all_features(df_val, baseline_ir, ['battery_temp'])
df_test = engineer_all_features(df_test, baseline_ir, ['battery_temp'])

try:
    with open('../models/feature_columns.json') as f: sel = json.load(f)
except: sel = [c for c in df_train.columns if c not in targets]

X_tr = df_train[[c for c in sel if c in df_train.columns]]
X_v = df_val[[c for c in sel if c in df_val.columns]]
y_temp_tr, y_temp_v = df_train['over_temp_flag'], df_val['over_temp_flag']
y_volt_tr, y_volt_v = df_train['over_voltage_flag'], df_val['over_voltage_flag']

scaler = StandardScaler()
Xtr = scaler.fit_transform(X_tr); Xv = scaler.transform(X_v)
print(f'X_train: {Xtr.shape}, X_val: {Xv.shape}')

## 1. over_voltage_flag — Feasibility Check

If fewer than 20 positive cases: classifier will overfit noise. A physics-based threshold is more reliable and defensible.

In [ ]:
# over_voltage_flag analysis
volt_pos = int(y_volt_tr.sum())
volt_total = len(y_volt_tr)
print(f'over_voltage_flag in Training Set:')
print(f'  Positive cases: {volt_pos} ({volt_pos/volt_total*100:.2f}%)')
print(f'  Negative cases: {volt_total-volt_pos} ({(volt_total-volt_pos)/volt_total*100:.2f}%)')

if volt_pos < 20:
    print(f'\n⚠️ Only {volt_pos} positive examples — INSUFFICIENT for meaningful classifier.')
    print('\n→ DECISION: Use rule-based fallback:')
    print('  Flag = 1 if action_voltage > 4.15 OR terminal_voltage > 4.18')
    print('\nThis is professional engineering judgment. Training on <20 positives')
    print('produces an overfit model worse than a domain-knowledge threshold.')
    use_voltage_rule = True
else:
    print('\nSufficient positives — will train a classifier.')
    use_voltage_rule = False

## 2. over_temp_flag — Class Balance Analysis

In [ ]:
# Class distribution
neg, pos = int((y_temp_tr==0).sum()), int((y_temp_tr==1).sum())
print(f'over_temp_flag Training Distribution:')
print(f'  Class 0 (Normal): {neg} ({neg/len(y_temp_tr)*100:.1f}%)')
print(f'  Class 1 (Over-temp): {pos} ({pos/len(y_temp_tr)*100:.1f}%)')
print(f'  Imbalance ratio: {neg/max(pos,1):.1f}:1')

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_temp_tr)
class_weights = {0: cw[0], 1: cw[1]}
print(f'\nBalanced class weights: class_0={cw[0]:.3f}, class_1={cw[1]:.3f}')
print('\nUsing class_weight="balanced" instead of SMOTE because:')
print('  - SMOTE interpolates between samples → creates impossible temporal states')
print('  - class_weight penalises minority misclassification without fake data')

## 3. Logistic Regression (Baseline)

**Theory**: Lecture 04 — Sigmoid outputs probability 0-1. Decision boundary y=1 if h(x)≥threshold. Using class_weight='balanced' adjusts for imbalance.

In [ ]:
# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(Xtr, y_temp_tr)
probs_lr = lr.predict_proba(Xv)[:,1]

# Threshold tuning
print('Threshold tuning for Logistic Regression:')
print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
best_f1, best_th_lr = 0, 0.5
for th in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55]:
    yp = (probs_lr >= th).astype(int)
    p, r, f = precision_score(y_temp_v,yp), recall_score(y_temp_v,yp), f1_score(y_temp_v,yp)
    marker = ' ✓' if r >= 0.80 and f > best_f1 else ''
    print(f'{th:>10.2f} {p:>10.4f} {r:>10.4f} {f:>10.4f}{marker}')
    if r >= 0.80 and f > best_f1: best_f1, best_th_lr = f, th

yp_lr = (probs_lr >= best_th_lr).astype(int)
print(f'\nSelected threshold: {best_th_lr}')
print(f'\nClassification Report:')
print(classification_report(y_temp_v, yp_lr, target_names=['Normal','Over-Temp']))

In [ ]:
# Confusion matrix for LR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_temp_v, yp_lr, display_labels=['Normal','Over-Temp'],
    cmap='Blues', ax=axes[0])
axes[0].set_title('Logistic Regression — Confusion Matrix', fontweight='bold')

# Annotate error types
axes[1].axis('off')
axes[1].text(0.1, 0.8, 'Confusion Matrix Interpretation:', fontsize=13, fontweight='bold')
axes[1].text(0.1, 0.65, '• TN (top-left): Correctly predicted Normal', fontsize=11)
axes[1].text(0.1, 0.50, '• FP (bottom-left): False Alarm — Type 1 Error', fontsize=11, color='#FF9800')
axes[1].text(0.1, 0.35, '• FN (top-right): Missed Over-Temp — Type 2 Error', fontsize=11, color='#F44336')
axes[1].text(0.1, 0.20, '• TP (bottom-right): Correctly detected Over-Temp', fontsize=11, color='#4CAF50')
axes[1].text(0.1, 0.05, '⚠️ FN is most dangerous: missed overheating event!', fontsize=11, fontweight='bold', color='red')
plt.tight_layout()
plt.savefig('../plots/cls_lr_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SVM with RBF Kernel

**Theory**: Lecture 06 — SVM finds optimal hyperplane maximising margin. RBF kernel maps to infinite-dimensional space — ideal when the boundary between safe/dangerous battery states is complex and non-linear.

**Why RBF**: Whether a battery overheats is not linearly separable in voltage-current-temperature space.

In [ ]:
# SVM with RBF kernel — Grid Search
print('Training SVM-RBF with grid search...')
best_svm, best_svm_f1 = None, 0
for C in [0.1, 1, 10, 100]:
    for gamma in ['scale', 'auto', 0.01, 0.1]:
        svm = SVC(kernel='rbf', C=C, gamma=gamma, class_weight='balanced',
                  probability=True, random_state=42)
        svm.fit(Xtr, y_temp_tr)
        yp = svm.predict(Xv)
        f = f1_score(y_temp_v, yp)
        r = recall_score(y_temp_v, yp)
        if f > best_svm_f1:
            best_svm, best_svm_f1 = svm, f
            print(f'  C={C}, gamma={gamma}: F1={f:.4f}, Recall={r:.4f} ✓')

probs_svm = best_svm.predict_proba(Xv)[:,1]
yp_svm = best_svm.predict(Xv)
print(f'\nBest SVM Results:')
print(classification_report(y_temp_v, yp_svm, target_names=['Normal','Over-Temp']))

## 5. XGBoost Classifier

**Theory**: Lecture 07 — XGBoost with `scale_pos_weight` handles imbalance. Early stopping prevents overfitting. This is the primary model candidate.

In [ ]:
# XGBoost Classifier
tscv = TimeSeriesSplit(n_splits=5)
scale_pw = neg / max(pos, 1)
print(f'scale_pos_weight = {scale_pw:.2f}')

xgb_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_alpha': [0, 0.5, 1.0]
}

gs_xgb = GridSearchCV(
    XGBClassifier(scale_pos_weight=scale_pw, eval_metric='logloss',
                  random_state=42, verbosity=0),
    xgb_grid, cv=tscv, scoring='f1', n_jobs=-1
)
gs_xgb.fit(Xtr, y_temp_tr)
print(f'Best params: {gs_xgb.best_params_}')

# Retrain with early stopping
xgb_cls = XGBClassifier(**gs_xgb.best_params_, scale_pos_weight=scale_pw,
    eval_metric='logloss', early_stopping_rounds=30, random_state=42, verbosity=0)
xgb_cls.fit(Xtr, y_temp_tr, eval_set=[(Xv, y_temp_v)], verbose=False)

probs_xgb = xgb_cls.predict_proba(Xv)[:,1]

In [ ]:
# Threshold tuning for XGBoost — targeting Recall >= 0.85, Precision >= 0.60
print('XGBoost Threshold Tuning:')
print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
best_f1x, best_thx = 0, 0.5
for th in np.arange(0.20, 0.70, 0.05):
    yp = (probs_xgb >= th).astype(int)
    p, r, f = precision_score(y_temp_v,yp), recall_score(y_temp_v,yp), f1_score(y_temp_v,yp)
    marker = ''
    if r >= 0.85 and p >= 0.60 and f > best_f1x:
        best_f1x, best_thx = f, th; marker = ' ← BEST'
    elif r >= 0.85 and f > best_f1x:
        best_f1x, best_thx = f, th; marker = ' ✓'
    print(f'{th:>10.2f} {p:>10.4f} {r:>10.4f} {f:>10.4f}{marker}')

yp_xgb = (probs_xgb >= best_thx).astype(int)
print(f'\nSelected threshold: {best_thx:.2f}')
print(f'\nClassification Report:')
print(classification_report(y_temp_v, yp_xgb, target_names=['Normal','Over-Temp']))

## 6. ROC Curves Comparison

**Theory**: Lecture 03 — ROC plots TPR vs FPR at various thresholds. AUC=1.0 is perfect, AUC=0.5 is random.

In [ ]:
# ROC comparison
fig, ax = plt.subplots(figsize=(10, 8))
RocCurveDisplay.from_estimator(lr, Xv, y_temp_v, ax=ax, name='Logistic Regression', color='#2196F3', linewidth=2)
RocCurveDisplay.from_estimator(best_svm, Xv, y_temp_v, ax=ax, name='SVM-RBF', color='#4CAF50', linewidth=2)
RocCurveDisplay.from_estimator(xgb_cls, Xv, y_temp_v, ax=ax, name='XGBoost', color='#FF9800', linewidth=2)
ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.set_title('ROC Curves — Over-Temperature Classification', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/roc_comparison_overtemp.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Precision-Recall curve with selected threshold marked
fig, ax = plt.subplots(figsize=(10, 8))
PrecisionRecallDisplay.from_estimator(lr, Xv, y_temp_v, ax=ax, name='LogReg', color='#2196F3')
PrecisionRecallDisplay.from_estimator(best_svm, Xv, y_temp_v, ax=ax, name='SVM', color='#4CAF50')
PrecisionRecallDisplay.from_estimator(xgb_cls, Xv, y_temp_v, ax=ax, name='XGBoost', color='#FF9800')

# Mark the selected threshold
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_temp_v, probs_xgb)
thr_idx = np.argmin(np.abs(thresholds - best_thx))
ax.plot(recalls[thr_idx], precisions[thr_idx], 'r*', markersize=20, label=f'Selected threshold ({best_thx:.2f})')
ax.set_title('Precision-Recall Curves', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# XGBoost confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_temp_v, yp_xgb, display_labels=['Normal','Over-Temp'],
    cmap='Oranges', ax=ax)
ax.set_title(f'XGBoost Confusion Matrix (threshold={best_thx:.2f})', fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/cls_xgb_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Comparison Summary

In [ ]:
# Results comparison
results = {}
for name, probs, yp, model in [
    ('LogisticRegression', probs_lr, yp_lr, lr),
    ('SVM-RBF', probs_svm, yp_svm, best_svm),
    ('XGBoost', probs_xgb, yp_xgb, xgb_cls)
]:
    results[name] = {
        'Recall': recall_score(y_temp_v, yp),
        'Precision': precision_score(y_temp_v, yp),
        'F1': f1_score(y_temp_v, yp),
        'ROC_AUC': roc_auc_score(y_temp_v, probs)
    }

results_df = pd.DataFrame(results).T
print('=== Classification Model Comparison ===')
print(results_df.to_string())

# Visual comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
models = list(results.keys()); colors = ['#2196F3','#4CAF50','#FF9800']
for ax, metric in zip(axes, ['Recall','Precision','F1','ROC_AUC']):
    vals = [results[m][metric] for m in models]
    bars = ax.bar(models, vals, color=colors)
    ax.set_title(metric, fontweight='bold'); ax.set_ylim(0, 1.1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.3f}', ha='center', fontsize=9)
plt.suptitle('Classification Model Comparison', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../plots/classification_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Best Classifier

In [ ]:
# Select best classifier (highest F1 with Recall >= 0.85)
valid = {k:v for k,v in results.items() if v['Recall'] >= 0.85}
if not valid: valid = results  # fallback
best_cls_name = max(valid, key=lambda k: valid[k]['F1'])
thresholds = {'LogisticRegression': best_th_lr, 'SVM-RBF': 0.5, 'XGBoost': float(best_thx)}
best_cls_model = {'LogisticRegression': lr, 'SVM-RBF': best_svm, 'XGBoost': xgb_cls}[best_cls_name]

print(f'Best classifier: {best_cls_name}')
print(f'  Recall: {results[best_cls_name]["Recall"]:.4f}')
print(f'  F1:     {results[best_cls_name]["F1"]:.4f}')
print(f'  Threshold: {thresholds[best_cls_name]}')

joblib.dump(best_cls_model, '../models/classification_model_temp.pkl')
print('\nSaved: models/classification_model_temp.pkl')

config = {
    'temp_threshold': thresholds[best_cls_name],
    'voltage_rule': 'action_voltage > 4.15 OR terminal_voltage > 4.18',
    'best_classifier': best_cls_name
}
with open('../models/classification_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved: models/classification_config.json')

with open('../models/classification_results.json', 'w') as f:
    json.dump({k:{mk:float(mv) for mk,mv in v.items()} for k,v in results.items()}, f, indent=2)
print('Saved: models/classification_results.json')